In [3]:
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils .data import DataLoader
from torchvision import datasets, transforms
from torchvision import models

DATA_ROOT = Path(r"D:\dataset\sampled_500")

TEST_DIR = DATA_ROOT / "val"
MODEL_PATH = "inat_resnet50_500classes_imagenet.pth"
MODEL_PATH = "inat_resnet50_500_ptclasses_imagenet.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [4]:
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=False
)

classes = checkpoint["classes"]
num_classes = checkpoint["num_classes"]

print("Number of classes:", num_classes)
print("Classes:", classes)

Number of classes: 500
Classes: ['00001_Animalia_Annelida_Polychaeta_Sabellida_Sabellidae_Sabella_spallanzanii', '00003_Animalia_Annelida_Polychaeta_Sabellida_Serpulidae_Spirobranchus_cariniferus', '00018_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Argiope_bruennichi', '00024_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Cyclosa_turbinata', '00038_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Neoscona_crucifera', '00055_Animalia_Arthropoda_Arachnida_Araneae_Filistatidae_Kukulcania_hibernalis', '00128_Animalia_Arthropoda_Arachnida_Araneae_Thomisidae_Synema_globosum', '00145_Animalia_Arthropoda_Arachnida_Opiliones_Phalangiidae_Phalangium_opilio', '00159_Animalia_Arthropoda_Chilopoda_Scolopendromorpha_Scolopendridae_Scolopendra_heros', '00203_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Cicindela_hirticollis', '00216_Animalia_Arthropoda_Insecta_Coleoptera_Carabidae_Scaphinotus_angusticollis', '00230_Animalia_Arthropoda_Insecta_Coleoptera_Cerambycidae_Knulliana_cincta', '00234_

In [5]:
test_transform = transforms.Compose([
    transforms.Resize(
        352,
        interpolation=transforms.InterpolationMode.BILINEAR,
        antialias=True
    ),
    transforms.CenterCrop(320),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

assert test_dataset.classes == classes, (
    "different testing set"
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Test images:", len(test_dataset))

Test images: 5000


In [6]:
model = models.resnet50(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [7]:
correct = 0
total = 0

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

test_accuracy = correct / total

print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Correct predictions: {correct}/{total}")

Test accuracy: 0.7944
Correct predictions: 3972/5000
